# blender_pipeline_v2 — تست Integration روی GPU

این notebook تست کامل pipeline را روی GPU واقعی انجام می‌دهد:
1. Clone ریپو از GitHub
2. نصب TripoSR و وابستگی‌ها
3. اجرای pipeline روی یک عکس واقعی
4. بررسی خروجی: mesh.obj و scene.py

> **مهم:** Runtime → Change runtime type → T4 GPU را انتخاب کن.

## سلول ۰ — بررسی GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: GPU ندارد — اجرا روی CPU خیلی کند خواهد بود')

## سلول ۱ — Clone ریپو

In [ ]:
!git clone https://github.com/nasrialireza959-pixel/blender_pipeline.git
%cd blender_pipeline

## سلول ۲ — نصب وابستگی‌ها و TripoSR

In [ ]:
!pip install -q rembg trimesh omegaconf pyyaml pillow
!python scripts/setup.py
print('نصب تمام شد.')

## سلول ۳ — دانلود عکس نمونه

In [ ]:
import urllib.request
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# عکس نمونه — یک ماگ ساده (object با پس‌زمینه سفید، مناسب برای TripoSR)
img_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/45/A_small_cup_of_coffee.JPG/640px-A_small_cup_of_coffee.JPG'
img_path = Path('examples/sample_images/test_input.jpg')
img_path.parent.mkdir(parents=True, exist_ok=True)

urllib.request.urlretrieve(img_url, img_path)

img = Image.open(img_path)
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title('عکس ورودی')
plt.axis('off')
plt.show()
print('اندازه:', img.size)

## سلول ۴ — اجرای pipeline

In [ ]:
import sys
sys.path.insert(0, 'src')

from core.config import load_config
from orchestration.pipeline import Pipeline

config = load_config('configs/default.yaml')
pipeline = Pipeline(config)

print('شروع pipeline...')
job = pipeline.run('examples/sample_images/test_input.jpg', object_name='CoffeeMug')

print(f'\nوضعیت: {job.status}')
print(f'Job ID: {job.id}')
print(f'mesh: {job.mesh_path}')
print(f'script: {job.blender_script_path}')

## سلول ۵ — بررسی خروجی mesh

In [ ]:
import trimesh

mesh = trimesh.load(str(job.mesh_path))
print('تعداد vertices:', len(mesh.vertices))
print('تعداد faces:', len(mesh.faces))
print('اندازه فایل:', round(job.mesh_path.stat().st_size / 1024, 1), 'KB')

assert len(mesh.vertices) > 0, 'mesh خالی است!'
assert len(mesh.faces) > 0, 'mesh face ندارد!'
print('\nمش سالم است.')

## سلول ۶ — بررسی اسکریپت Blender

In [ ]:
script = job.blender_script_path.read_text(encoding='utf-8')

assert 'bpy.ops.wm.obj_import' in script, 'API مدرن Blender 4.x نیست!'
assert 'import_scene.obj' not in script, 'API منسوخ پیدا شد!'
assert 'CoffeeMug' in script, 'نام object در اسکریپت نیست!'

print('اسکریپت Blender تایید شد.')
print('\n--- ۲۰ خط اول اسکریپت ---')
print('\n'.join(script.splitlines()[:20]))

## سلول ۷ — بررسی job.json

In [ ]:
import json

metadata = json.loads(job.metadata_path.read_text(encoding='utf-8'))
print(json.dumps(metadata, ensure_ascii=False, indent=2))

assert metadata['status'] == 'done', f"وضعیت اشتباه: {metadata['status']}"
assert metadata['error'] is None, f"خطا ثبت شده: {metadata['error']}"
print('\nجمع‌بندی: pipeline با موفقیت اجرا شد.')

## سلول ۸ — اجرای unit tests (تایید نهایی)

In [ ]:
!python -m pytest tests/ -v